# SMS Spam Detection - ML Lab

**Dataset:** SMS Spam Collection (~5,500 messages)

**Goal:** Classify SMS messages as **Spam** or **Ham** (legitimate).

**What you will learn:**
- How to clean and preprocess text data
- How to convert text to numbers using TF-IDF
- How to train and compare multiple ML models
- How to evaluate models with proper metrics

---
## Step 0: Library Guide - What we use and why

### Libraries and their alternatives

| Library | Purpose | Alternative | Why this one |
|---|---|---|---|
| **Pandas** | Read and manipulate tabular data | Polars, Dask | Most popular, easy to use |
| **NumPy** | Work with numeric arrays | JAX, CuPy | Foundation of all ML libraries |
| **Matplotlib** | Basic plotting | Plotly, Bokeh | Most control over plots |
| **Seaborn** | Beautiful statistical plots | Plotly Express | Easy wrapper over Matplotlib |
| **NLTK** | Text processing (stopwords, stemming) | spaCy, TextBlob | Great for learning |
| **WordCloud** | Visualize frequent words | None (unique) | Quick visual insight |
| **Scikit-learn** | ML models, features, metrics | PyCaret, XGBoost | Most widely used |

### Text Processing Components

| Component | What it does | Example |
|---|---|---|
| **Lowercase** | Convert all letters to small | "FREE" -> "free" |
| **Stopwords** | Remove common words | "the", "is", "and" removed |
| **Stemming** | Reduce word to root form | "calling" -> "call" |
| **TF-IDF** | Convert text to numbers | "free" -> 0.85 (important) |

---
## Step 1: Import Libraries

Import all the tools we need first.

In [ ]:
# --- Data Handling ---
import pandas as pd      # For reading tabular data (like Excel)
import numpy as np       # For working with numeric arrays

# --- Visualization ---
import matplotlib.pyplot as plt  # For plotting
import seaborn as sns            # For beautiful statistical plots
%matplotlib inline
sns.set_style('whitegrid')  # Set plot background

# --- Text Processing ---
import re                           # Regular expressions (remove punctuation)
import nltk                         # Natural Language Processing library
nltk.download('stopwords')          # Download stopwords (the, is, and etc.)
nltk.download('punkt')              # For tokenization
nltk.download('punkt_tab')          # Newer version of punkt
from nltk.corpus import stopwords   # Stopword list
from nltk.stem import PorterStemmer # For stemming (calling -> call)

# --- WordCloud ---
from wordcloud import WordCloud     # To see which words are most used

# --- Machine Learning ---
from sklearn.model_selection import train_test_split    # Split data
from sklearn.feature_extraction.text import TfidfVectorizer  # Convert text to numbers
from sklearn.preprocessing import LabelEncoder           # Convert labels to numbers (ham=0, spam=1)
from sklearn.naive_bayes import MultinomialNB           # Naive Bayes model
from sklearn.linear_model import LogisticRegression     # Logistic Regression
from sklearn.svm import SVC                             # Support Vector Machine
from sklearn.tree import DecisionTreeClassifier        # Decision Tree
from sklearn.ensemble import RandomForestClassifier    # Random Forest
from sklearn.neighbors import KNeighborsClassifier     # K-Nearest Neighbors
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

---
## Step 2: Load Dataset

We use **Pandas** to read the CSV file. The dataset has two columns:
- **label** - `ham` (legitimate) or `spam`
- **message** - the SMS text

We remove duplicates because same messages appearing multiple times will give wrong model performance.

In [ ]:
# Read CSV from URL (using Pandas)
url = "https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv"
df = pd.read_csv(url, encoding='latin-1')  # latin-1 encoding needed for special characters

df = df[['v1', 'v2']]           # Keep only needed columns
df.columns = ['label', 'message'] # Rename columns nicely
df = df.drop_duplicates().reset_index(drop=True)  # Remove duplicates

print(f"Total messages: {df.shape[0]}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
df.head()

---
## Step 3: Exploratory Data Analysis (EDA)

Before building models, we explore the data:
1. **Message length** - Are spam messages longer than ham?
2. **Class balance** - How many ham vs spam?
3. **WordCloud** - Which words are used most?

In [ ]:
# --- 1. Character length and word count ---
df['length'] = df['message'].apply(len)  # Character count per message
df['word_count'] = df['message'].apply(lambda x: len(nltk.word_tokenize(str(x))))  # Word count

print("--- Average length: Ham vs Spam ---")
print(df.groupby('label')[['length', 'word_count']].mean())  # Average length and word count

# --- 2. Message length distribution (Histogram) ---
plt.figure(figsize=(12, 5))

# Ham message length
plt.subplot(1, 2, 1)  # 1st of 2 plots in 1 row
sns.histplot(df[df['label'] == 'ham']['length'], bins=50, color='green')
plt.title('Ham Message Length')
plt.xlabel('Length (characters)')

# Spam message length
plt.subplot(1, 2, 2)  # 2nd of 2 plots in 1 row
sns.histplot(df[df['label'] == 'spam']['length'], bins=50, color='red')
plt.title('Spam Message Length')
plt.xlabel('Length (characters)')

plt.tight_layout()
plt.show()

# --- 3. Class distribution ---
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='label', palette='Set2')
plt.title('Ham vs Spam Count')
plt.show()

---
## Step 4: WordCloud - Which words are most used?

**WordCloud** shows which words appear most frequently. Bigger word = more frequent.

- **Ham WordCloud** - normal conversation words (ok, come, got, etc.)
- **Spam WordCloud** - promotional words (free, win, prize, call, etc.)

In [ ]:
# Join all ham messages together
ham_words = " ".join(df[df['label'] == 'ham']['message'])
# Join all spam messages together
spam_words = " ".join(df[df['label'] == 'spam']['message'])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Ham WordCloud
wordcloud_ham = WordCloud(width=600, height=400, background_color='white', colormap='Blues').generate(ham_words)
axes[0].imshow(wordcloud_ham, interpolation='bilinear')
axes[0].axis('off')  # Hide axes
axes[0].set_title('Most Frequent Words - HAM', fontsize=14)

# Spam WordCloud
wordcloud_spam = WordCloud(width=600, height=400, background_color='white', colormap='Reds').generate(spam_words)
axes[1].imshow(wordcloud_spam, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Most Frequent Words - SPAM', fontsize=14)

plt.tight_layout()
plt.show()

---
## Step 5: Text Cleaning

Raw text is messy. We clean it in 4 steps:

| Step | What we do | Example | Why needed |
|---|---|---|---|
| **Lowercase** | Convert all to small letters | "FREE" -> "free" | "FREE" and "free" are same word |
| **Remove punctuation** | Keep only letters | "free!!!" -> "free" | Punctuation marks not needed |
| **Remove stopwords** | Filter common words | "the", "is", "and" removed | These words have no meaning |
| **Stemming** | Reduce to root form | "calling" -> "call" | "calling" and "call" mean same |

In [ ]:
# Load stopwords (the, is, at, which, on etc.)
stop_words = set(stopwords.words('english'))
# Create Porter Stemmer (reduces words to root form)
stemmer = PorterStemmer()

def clean_text(text):
    """
    Text cleaning function:
    1. Convert to lowercase
    2. Keep only alphabets and spaces
    3. Remove stopwords
    4. Apply stemming
    """
    text = text.lower()                          # 1. Lowercase
    text = re.sub(r'[^a-z\s]', ' ', text)         # 2. Keep only a-z and spaces
    words = text.split()                          # 3. Split into words (tokenize)
    words = [w for w in words if w not in stop_words]  # 4. Remove stopwords
    words = [stemmer.stem(w) for w in words]     # 5. Stemming
    return " ".join(words)                        # 6. Join back together

df['clean_message'] = df['message'].apply(clean_text)

# Show before and after cleaning
print("--- Before and After Cleaning ---")
df[['message', 'clean_message']].head(10)

---
## Step 6: Feature Extraction (TF-IDF)

**ML models understand numbers, not text.** So we need to convert text to numbers.

**What is TF-IDF?**
- **TF (Term Frequency)** - How often a word appears in a message
- **IDF (Inverse Document Frequency)** - If a word appears in ALL messages, its importance is low

**Example:**
- "free" appears more in Spam messages -> IDF gives higher score -> Important
- "the" appears in all messages -> IDF gives lower score -> Not important

**Alternative:** `CountVectorizer` (Bag-of-Words) - only counts, does not consider importance.

In [ ]:
# Create TF-IDF vectorizer (only top 3000 words)
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['clean_message']).toarray()  # Convert text to numbers

# Convert labels to numbers: ham=0, spam=1
le = LabelEncoder()
y = le.fit_transform(df['label'])

print(f"Feature matrix: {X.shape}  (each message = 3000 numbers)")
print(f"Labels: {le.classes_} -> {le.transform(le.classes_)}")

---
## Step 7: Train-Test Split

Split data into:
- **80% training** - model learns from this
- **20% testing** - evaluate on unseen data

**`stratify=y`** ensures same ham/spam ratio in both sets.

**Why split?** If we train on all data, model will memorize, not learn. Separate test data shows how model performs on new data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {X_train.shape[0]} samples")
print(f"Testing:  {X_test.shape[0]} samples")

---
## Step 8: Train Multiple Models

We train **6 different algorithms** on the same data and compare:

| Algorithm | Type | Why use it |
|---|---|---|
| **Naive Bayes** | Probability-based | Classic choice for text |
| **Logistic Regression** | Linear | Fast, easy to interpret |
| **SVM** | Margin-based | Good for high-dimensional data |
| **Decision Tree** | Rule-based | Easy to understand |
| **Random Forest** | Ensemble | Reduces overfitting |
| **KNN** | Similarity-based | Simple nearest neighbor search |

In [ ]:
# Create 6 models
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear', probability=True),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

results = []       # Store results
trained_models = {} # Store trained models
predictions = {}    # Store predictions

for name, model in models.items():
    model.fit(X_train, y_train)  # Train the model
    y_pred = model.predict(X_test)  # Predict on test data

    # Collect metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })
    trained_models[name] = model
    predictions[name] = y_pred

# Results table (best model on top)
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
results_df

---
## Step 9: Model Comparison

**What the metrics mean:**
- **Accuracy** - Overall correct percentage
- **Precision** - Of predicted spam, how many are actually spam? (low = false alarms)
- **Recall** - Of actual spam, how many did we catch? (low = missed spam)
- **F1-Score** - Balance of precision and recall

**Important for spam detection:** Blocking a real message (False Positive) is worse than letting spam through.

In [ ]:
# Bar chart comparison
results_melted = results_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(11, 6))
sns.barplot(data=results_melted, x="Model", y="Score", hue="Metric")
plt.title("Model Comparison")
plt.xticks(rotation=25)
plt.ylim(0.8, 1.0)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Best model details
best_model_name = results_df.iloc[0]['Model']  # Best model name
print(f"Best model: {best_model_name}\n")

# Confusion matrix
cm = confusion_matrix(y_test, predictions[best_model_name])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Classification report
print("\n--- Classification Report ---")
print(classification_report(y_test, predictions[best_model_name], target_names=['Ham', 'Spam']))

---
## Step 10: Predict New Messages

We use the best model to classify brand-new messages. This simulates real-world usage.

**Process:**
1. Clean the new message (clean_text)
2. Convert to numbers using TF-IDF
3. Predict using the model
4. Convert number back to text (Ham/Spam)

In [ ]:
def predict_message(text):
    """Predict Spam/Ham for new message"""
    cleaned = clean_text(text)                        # 1. Clean
    vec = tfidf.transform([cleaned]).toarray()        # 2. Convert to numbers
    pred = trained_models[best_model_name].predict(vec)[0]  # 3. Predict
    return le.inverse_transform([pred])[0]            # 4. Number -> Text

# Test on new messages
sample_messages = [
    "Congratulations! You've won a $1000 Walmart gift card. Click here to claim now!!!",
    "Hey, are we still on for lunch tomorrow at 1pm?",
    "URGENT: Your bank account has been suspended. Verify your details immediately.",
    "Don't forget to bring your laptop to class tomorrow.",
    "FREE entry into our weekly competition, text WIN to 80086 now!"
]

print("="*60)
for msg in sample_messages:
    print(f"[{predict_message(msg).upper():5}]  {msg}")
print("="*60)

---
## Key Takeaways

1. **Text preprocessing matters** - cleaning removes noise and improves accuracy
2. **TF-IDF is powerful** - converts text to meaningful numeric features
3. **Compare multiple models** - no single algorithm is always best
4. **Precision vs Recall trade-off** - for spam: missing spam (low recall) vs blocking real messages (low precision)

### Try these extensions:
- Try `CountVectorizer` (Bag-of-Words) instead of TF-IDF
- Add n-grams: `ngram_range=(1,2)` in the vectorizer
- Use `GridSearchCV` to tune hyperparameters
- Try a simple LSTM model with Keras